# Churn

Compare logo churn and revenue churn from the Tidemill engine with
ground-truth values computed directly from Stripe subscription lifecycle data.

## How churn is measured

**Logo (customer) churn** — the fraction of *customers* who fully cancel:

```
logo_churn = customers_fully_churned / customers_active_at_period_start
```

A customer is "active at start" if they had at least one subscription created
before the period start that was not yet canceled.  They are "fully churned"
if, by the period end, they have zero active subscriptions remaining.

**Revenue churn** — the fraction of *MRR* lost to cancellations:

```
revenue_churn = |churned_MRR| / starting_MRR
```

`churned_MRR` sums the `mrr_cents` of every subscription whose `canceled_at`
timestamp falls within `[start, end)`.

## Stripe fields used

- `subscription.status` — "canceled" means the sub ended
- `subscription.canceled_at` — Unix timestamp of cancellation
- `subscription.created` — Unix timestamp of creation
- `subscription.items.data[*].price` — used to compute per-sub MRR

The denominator matters: both Tidemill and this notebook use the same
starting MRR (from Tidemill's waterfall) so the comparison is apples-to-apples.

In [12]:
import os

import stripe

from tidemill import reports
from tidemill.reports.stripecheck import StripeData, TidemillClient

stripe.api_key = os.environ["STRIPE_API_KEY"]
reports.setup()

START, END = "2025-08-01", "2026-03-31"
CHURN_START = "2025-08-01"  # need a prior month as baseline denominator
tm = TidemillClient()
sd = StripeData()

## 1. Churn comparison — Tidemill vs Stripe

`stripecheck.compare.churn()` queries both sources over the same window
and uses the same starting-MRR denominator for revenue churn.

In [13]:
data = reports.churn.stripe_overview(tm, sd, CHURN_START, END)
reports.churn.style_stripe_overview(data)

Metric,Tidemill,Stripe,Match
Logo churn,16.7%,23.5%,False
Revenue churn,16.2%,15.8%,False
Active at start,17,17,True
Fully churned,4,4,True


In [3]:
reports.churn.plot_stripe_overview(data)

## 2. Churn from Stripe (detailed)

`stripecheck.stripe_metrics.churn_rates()` implements the per-customer
analysis:

1. Group subscriptions by `customer` ID
2. For each customer, check if they had active subs at period start
3. Check if they still have active subs at period end
4. Sum MRR of subscriptions canceled within the window

The "active at start" check uses:
```python
created_at < start AND (canceled_at IS NULL OR canceled_at >= start)
```

In [4]:
from tidemill.reports.stripecheck.stripe_metrics import churn_rates

stripe_churn = churn_rates(sd.subscriptions, CHURN_START, END)
print("Stripe churn detail:")
for k, v in stripe_churn.items():
    if isinstance(v, float):
        print(f"  {k:25s}  {v * 100:.1f}%")
    else:
        print(f"  {k:25s}  {v}")

Stripe churn detail:
  logo_churn                 10.5%
  revenue_churn              4.9%
  active_at_start            19
  fully_churned              2
  churned_mrr_cents          4200
  starting_mrr_cents         86233


## 3. Churn visualization

In [5]:
reports.churn.plot_stripe_detail(stripe_churn)

## 4. Per-customer churn detail — Stripe vs Tidemill

Two tables show exactly which customers feed into each side's churn
calculation, followed by a merged diff that flags any discrepancies.

**Stripe** groups subscriptions by customer, checks `created_at < start`
and `canceled_at IS NULL OR >= start` to determine "active at start",
then sums `mrr_cents` of subscriptions canceled in `[start, end)`.

**Tidemill** tracks customer state via `metric_churn_customer_state`
(incremented on `subscription.activated`, decremented on
`subscription.churned`) and records per-event churn amounts in
`metric_churn_event`.

In [6]:
stripe_detail = reports.churn.stripe_customer_detail(sd, CHURN_START, END)
reports.churn.style_customer_detail(stripe_detail, "Stripe")

customer,active_at_start,fully_churned,subs_at_start,starting_mrr,churned_mrr
cus_ULWwH5W4Rtwl58,1,0,1,$21.00,$0.00
cus_ULWwJAvHyIN6uT,1,0,1,$21.00,$0.00
cus_ULWwNaOa2hh32z,1,0,1,$79.00,$0.00
cus_ULWwO9uqpSPVUt,1,0,1,$79.00,$0.00
cus_ULWwQexwEAjgi3,1,0,1,$79.00,$0.00
cus_ULWwTgCTM36v4e,1,0,1,$207.50,$0.00
cus_ULWwaIAUzeOUbH,1,0,1,$65.83,$0.00
cus_ULWwaKJiAP4F85,1,0,1,$79.00,$0.00
cus_ULWwfk0VwelKjT,1,0,1,$21.00,$0.00
cus_ULWwrTRn9Nc2KQ,1,0,1,$21.00,$0.00


In [7]:
tm_detail = reports.churn.tidemill_customer_detail(tm, CHURN_START, END)
reports.churn.style_customer_detail(tm_detail, "Tidemill")

customer,active_at_start,fully_churned,starting_mrr,churned_mrr
cus_ULWwH5W4Rtwl58,1,0,$21.00,$0.00
cus_ULWwJAvHyIN6uT,1,0,$21.00,$0.00
cus_ULWwNaOa2hh32z,1,0,$79.00,$0.00
cus_ULWwO9uqpSPVUt,1,0,$79.00,$0.00
cus_ULWwQexwEAjgi3,1,0,$21.00,$0.00
cus_ULWwScgoS7pqiF,1,0,$79.00,$79.00
cus_ULWwTgCTM36v4e,1,0,$207.50,$0.00
cus_ULWwaIAUzeOUbH,1,0,$65.83,$0.00
cus_ULWwaKJiAP4F85,1,0,$79.00,$0.00
cus_ULWwfk0VwelKjT,1,0,$21.00,$0.00


In [8]:
diff = reports.churn.customer_churn_diff(stripe_detail, tm_detail)
reports.churn.style_customer_churn_diff(diff)

customer,s_fully_churned,t_fully_churned,s_starting_mrr,t_starting_mrr,s_churned_mrr,t_churned_mrr,notes
cus_ULWwH5W4Rtwl58,False,0,$21.00,$21.00,$0.00,$0.00,
cus_ULWwJAvHyIN6uT,False,0,$21.00,$21.00,$0.00,$0.00,
cus_ULWwNaOa2hh32z,False,0,$79.00,$79.00,$0.00,$0.00,
cus_ULWwO9uqpSPVUt,False,0,$79.00,$79.00,$0.00,$0.00,
cus_ULWwQexwEAjgi3,False,0,$79.00,$21.00,$0.00,$0.00,start MRR: $79 vs $21
cus_ULWwScgoS7pqiF,nan,0,$0.00,$79.00,$0.00,$79.00,Tidemill only
cus_ULWwTgCTM36v4e,False,0,$207.50,$207.50,$0.00,$0.00,
cus_ULWwaIAUzeOUbH,False,0,$65.83,$65.83,$0.00,$0.00,
cus_ULWwaKJiAP4F85,False,0,$79.00,$79.00,$0.00,$0.00,
cus_ULWwfk0VwelKjT,False,0,$21.00,$21.00,$0.00,$0.00,


## 5. Monthly Churn Rate Timeline

Compute logo and revenue churn rates per month to visualize churn trends over time.

In [9]:
churn_df = reports.churn.timeline(tm, CHURN_START, END)
reports.churn.style_timeline(churn_df)

,logo_churn,revenue_churn
month,,
2025-11,4.8%,10.6%
2025-12,8.0%,5.0%
2026-01,7.7%,4.3%
2026-02,7.1%,4.2%


In [10]:
reports.churn.plot_timeline(churn_df)

## 6. Monthly Churned MRR

Lost MRR per month from the MRR waterfall — shows the dollar impact of churn over time.

In [11]:
reports.churn.plot_monthly_lost_mrr(reports.churn.monthly_lost_mrr(tm, START, END))